# HDT 1: Pandas, SQL y DuckDB

**Ciencia de Datos, Sección A** · Asignada: martes 28 de julio · **Entrega: martes 4 de agosto, 23:59**

**Nombre:** Milton Beltran

Completar las celdas marcadas con `# ¿Qué va aquí?`. Cada ejercicio incluye una verificación comentada: descomentar para comprobar el resultado. Antes de entregar: **Kernel → Restart & Run All** (un notebook que no corre de arriba a abajo pierde 0.5 pts).

AI: resolver sin AI. Si se usó para entender un concepto, anotarlo en la mini-bitácora del final.

## Setup

Si falta DuckDB: descomentar la línea de instalación, ejecutar la celda una vez y volver a comentarla.

In [1]:
# !uv add duckdb    (en terminal)  o descomentar:  %pip install duckdb
import pandas as pd
import duckdb

URL = ("https://raw.githubusercontent.com/"
       "mwaskom/seaborn-data/master/penguins.csv")
penguins = pd.read_csv(URL)

# Tabla de nombres científicos (para los JOIN)
especies = pd.DataFrame({
    "species": ["Adelie", "Chinstrap", "Gentoo"],
    "nombre_cientifico": ["Pygoscelis adeliae",
                          "Pygoscelis antarcticus",
                          "Pygoscelis papua"],
})

penguins.head()
# especies.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


## Parte A · Pandas (1.0 pt)

### Ejercicio 1 (0.10): cargar y explorar

Mostrar: (a) el `shape` del DataFrame, (b) los `dtypes`, y (c) cuántos nulos tiene **cada columna**.

In [2]:
# ¿Qué va aquí? (tres expresiones, una por inciso)


print(penguins.shape)
print("------------------------------------------------------------------------------------")
print("El dataset tiene", penguins.shape[0], "filas y", penguins.shape[1], "columnas")
print("------------------------------------------------------------------------------------")
print(penguins.dtypes)
print("------------------------------------------------------------------------------------")
print(penguins.isna().sum())




# Verificación: el dataset original tiene 344 filas y 7 columnas,
# y la columna sex es la que más nulos tiene (11).

(344, 7)
------------------------------------------------------------------------------------
El dataset tiene 344 filas y 7 columnas
------------------------------------------------------------------------------------
species                  str
island                   str
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                      str
dtype: object
------------------------------------------------------------------------------------
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


### Ejercicio 2 (0.15): limpieza mínima

Crear un DataFrame `limpio` **sin** las filas que tengan nulo en cualquier columna. Reportar con un `print` cuántas filas se perdieron respecto al original.

In [3]:
limpio =  penguins.dropna()  # ¿Qué va aquí?

print("Se eliminaron un total de",penguins.shape[0] - limpio.shape[0],"filas respecto al original")
# print(f"Se perdieron {...} filas")

# Verificación (descomentar):
assert limpio.shape[0] == 333 and limpio.isna().sum().sum() == 0

Se eliminaron un total de 11 filas respecto al original


### Ejercicio 3 (0.15): máscaras y orden

De `limpio`: los pingüinos de la isla **Biscoe** con masa corporal **mayor a 4500 g**, ordenados de mayor a menor masa. Mostrar solo las columnas `species`, `island`, `body_mass_g`.

In [4]:
mascara_limpio = (limpio["body_mass_g"] > 4500) & (limpio["island"] == "Biscoe")
filtros_limpio=["species","island","body_mass_g"]

pesados_biscoe = limpio.loc[mascara_limpio,filtros_limpio].sort_values("body_mass_g",ascending=False)  # ¿Qué va aquí?


pesados_biscoe




# Verificación (descomentar):
#assert (pesados_biscoe["island"] == "Biscoe").all()
#assert (pesados_biscoe["body_mass_g"] > 4500).all()
#assert pesados_biscoe["body_mass_g"].is_monotonic_decreasing

,species,island,body_mass_g
237,Gentoo,Biscoe,6300.0
253,Gentoo,Biscoe,6050.0
337,Gentoo,Biscoe,6000.0
297,Gentoo,Biscoe,6000.0
299,Gentoo,Biscoe,5950.0
...,...,...,...
248,Gentoo,Biscoe,4600.0
306,Gentoo,Biscoe,4600.0
296,Gentoo,Biscoe,4600.0
328,Gentoo,Biscoe,4575.0


### Ejercicio 4 (0.20): groupby con dos funciones

Masa corporal por **especie y sexo**: el **promedio** y el **conteo**, en una sola operación con `groupby` + `agg`.

In [5]:
categorias=["species","sex"]
resumen = limpio.groupby(categorias).agg(
    masa_corporal=("body_mass_g","mean"),
    conteo_total=("sex","count")
)
resumen

# Verificación: el grupo más pesado debe ser Gentoo macho (~5485 g en promedio).

masa_corporal  conteo_total
species   sex                                
Adelie    FEMALE    3368.835616            73
          MALE      4043.493151            73
Chinstrap FEMALE    3527.205882            34
          MALE      3938.970588            34
Gentoo    FEMALE    4679.741379            58
          MALE      5484.836066            61

### Ejercicio 5 (0.20): columna derivada

Agregar a `limpio` una columna `bill_ratio` = largo del pico / profundidad del pico. Mostrar el promedio de `bill_ratio` **por especie**, ordenado descendente. ¿Qué especie tiene el pico proporcionalmente más alargado?

In [6]:
# ¿Qué va aquí?
limpio["bill_ratio"] = limpio.bill_length_mm / limpio.bill_depth_mm

categorias=["species"]
resumen = limpio.groupby(categorias)["bill_ratio"].mean().sort_values(ascending=False)

resumen

# ¿Qué va aquí?

# Verificación: Gentoo debe quedar de primero (~3.2).

species
Gentoo       3.176602
Chinstrap    2.653756
Adelie       2.121478
Name: bill_ratio, dtype: float64

### Ejercicio 6 (0.20): merge

Unir `limpio` con la tabla `especies` para que cada fila tenga su `nombre_cientifico`. Mostrar una fila de cada especie para comprobar.

In [7]:
#limpio

#especies


con_nombres = pd.merge(limpio, especies, on="species",how="left")

#con_nombres["nombre_cientifico"] = con_nombres.groupby("species")["nombre_cientifico"].transform("count")



mergeado = con_nombres.drop_duplicates("species")[["species", "nombre_cientifico"]]

mergeado









# Verificación (descomentar):
# assert con_nombres.shape[0] == limpio.shape[0]
# assert "nombre_cientifico" in con_nombres.columns

,species,nombre_cientifico
0,Adelie,Pygoscelis adeliae
146,Chinstrap,Pygoscelis antarcticus
214,Gentoo,Pygoscelis papua


## Parte B · SQL con DuckDB (0.8 pt)

DuckDB consulta directamente los DataFrames en memoria: `duckdb.sql("SELECT ... FROM limpio")`. Cerrar cada consulta con `.df()` para ver el resultado como DataFrame.

### Ejercicio 7 (0.20): SELECT / WHERE / ORDER BY

El ejercicio 3, ahora en SQL: especie, isla y masa de los pingüinos de Biscoe con masa mayor a 4500 g, ordenados de mayor a menor.

In [8]:
q7 = """
SELECT species, island, body_mass_g
FROM limpio
WHERE island = 'Biscoe' AND body_mass_g > 4500
ORDER BY body_mass_g DESC
"""
duckdb.sql(q7).df()

# Verificación: debe dar las mismas filas que el ejercicio 3.

,species,island,body_mass_g
0,Gentoo,Biscoe,6300.0
1,Gentoo,Biscoe,6050.0
2,Gentoo,Biscoe,6000.0
3,Gentoo,Biscoe,6000.0
4,Gentoo,Biscoe,5950.0
...,...,...,...
101,Gentoo,Biscoe,4600.0
102,Gentoo,Biscoe,4600.0
103,Adelie,Biscoe,4600.0
104,Gentoo,Biscoe,4575.0


### Ejercicio 8 (0.20): GROUP BY + HAVING

Especies cuya masa corporal **promedio** supera los 4000 g, con su promedio redondeado.

In [9]:
q8 = """
SELECT species, ROUND(AVG(body_mass_g), 2) AS masa_promedio
FROM limpio
GROUP BY species
HAVING AVG(body_mass_g) > 4000
"""
duckdb.sql(q8).df()

# Verificación: solo una especie debe aparecer. ¿Cuál? Comparar con el resultado
# del ejercicio 4.

,species,masa_promedio
0,Gentoo,5092.44


### Ejercicio 9 (0.20): JOIN

El ejercicio 6, ahora en SQL: unir `limpio` con `especies` y mostrar especie, nombre científico y masa promedio por especie.

In [10]:
q9 = """
SELECT l.species, e.nombre_cientifico, ROUND(AVG(l.body_mass_g), 2) AS masa_promedio
FROM limpio AS l
JOIN especies AS e ON l.species = e.species
GROUP BY l.species, e.nombre_cientifico
ORDER BY masa_promedio DESC
"""
duckdb.sql(q9).df()

# Verificación: 3 filas, una por especie, cada una con su Pygoscelis.

,species,nombre_cientifico,masa_promedio
0,Gentoo,Pygoscelis papua,5092.44
1,Chinstrap,Pygoscelis antarcticus,3733.09
2,Adelie,Pygoscelis adeliae,3706.16


### Ejercicio 10 (0.20): window function

Los **3 pingüinos más pesados de cada especie**, usando `RANK() OVER (PARTITION BY ... ORDER BY ...)`. Pista de la sesión 3: la window function se calcula en una subconsulta y se filtra afuera.

In [11]:
q10 = """
SELECT species, island, sex, body_mass_g, puesto
FROM (
    SELECT species, island, sex, body_mass_g,
           RANK() OVER (PARTITION BY species ORDER BY body_mass_g DESC) AS puesto
    FROM limpio
)
WHERE puesto <= 3
ORDER BY species, puesto
"""
duckdb.sql(q10).df()

# Verificación: alrededor de 9 filas (3 por especie; puede haber empates),
# y el rango nunca debe pasar de 3.

,species,island,sex,body_mass_g,puesto
0,Adelie,Biscoe,MALE,4775.0,1
1,Adelie,Biscoe,MALE,4725.0,2
2,Adelie,Torgersen,MALE,4700.0,3
3,Chinstrap,Dream,MALE,4800.0,1
4,Chinstrap,Dream,MALE,4550.0,2
5,Chinstrap,Dream,MALE,4500.0,3
6,Gentoo,Biscoe,MALE,6300.0,1
7,Gentoo,Biscoe,MALE,6050.0,2
8,Gentoo,Biscoe,MALE,6000.0,3
9,Gentoo,Biscoe,MALE,6000.0,3


## Parte C · Criterio (0.2 pt)

### Ejercicio 11 (0.20)

Los mismos análisis se resolvieron en Pandas y en SQL. En 3-4 líneas, **con base en el trabajo de esta hoja** (no de memoria): ¿cuándo conviene cada herramienta? Mencionar al menos una operación que resultó más natural en cada una.



**Respuesta:** Pandas me sirvió mejor cuando el resultado se queda en el análisis. SQL conviene cuando lo que quiero es una pregunta puntual con filtro y agregación: el ejercicio 8 (GROUP BY + HAVING) y el ejercicio 10 (RANK con PARTITION BY) salieron casi como se leen en español, mientras que en Pandas el "top 3 por grupo" habría necesitado groupby + rank + máscara.

## Mini-bitácora de AI (opcional, no penaliza)

Si se usó AI para entender algún concepto, anotar aquí qué se preguntó y qué se entendió. Si no se usó, escribir "No se usó".

- ...

## Anexo: repaso de las sesiones 2 y 3

Regla de los ejercicios de NumPy: **sin ciclos `for`**.

In [12]:
import numpy as np

rng = np.random.default_rng(7)

### A1 (sesión 2): z-score sin loops

Normalizar un array: restar la media y dividir entre la desviación estándar.

In [13]:
alturas = rng.normal(170, 10, size=1000)

def z_score(x):
    # sin for: la resta y la división se aplican elemento por elemento
    return (x - x.mean()) / x.std()

z = z_score(alturas)
# Verificación (descomentar):
print(round(z.mean(), 4), round(z.std(), 4))  # ~0 y ~1

-0.0 1.0


### A2 (sesión 2): distancias con broadcasting

Distancia euclidiana de cada punto a un centro, sin loops.

In [14]:
puntos = rng.normal(size=(500, 2))   # 500 puntos en 2D
centro = np.array([1.0, 1.0])

# Pista: (puntos - centro) usa broadcasting (500,2) - (2,)
# Luego: elevar al cuadrado, sumar con axis=1, sacar raíz
diferencias = puntos - centro
distancias = np.sqrt((diferencias ** 2).sum(axis=1))

# ¿Cuántos puntos están a menos de 1 del centro?
cercanos = (distancias < 1).sum()

# Verificación (descomentar):
print(distancias.shape)  # (500,)
print(cercanos)

(500,)
86


### A3 (sesión 3): propinas por día y turno

Dataset `tips` (propinas de un restaurante). `pct` = propina como fracción de la cuenta.

In [15]:
URL_TIPS = ("https://raw.githubusercontent.com/"
            "mwaskom/seaborn-data/master/tips.csv")
tips = pd.read_csv(URL_TIPS)
tips["pct"] = tips["tip"] / tips["total_bill"]
tips.head()

,total_bill,tip,sex,smoker,day,time,size,pct
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808


In [16]:
# a) porcentaje medio de propina por dia
# b) por dia Y turno (day, time), en una tabla
# c) el dia con el mayor porcentaje medio
# d) numero de mesas por dia

resumen = tips.groupby("day").agg(
    pct_promedio=("pct","mean"),
    num_mesas=("pct","count")
).sort_values("pct_promedio",ascending=False)

dia_turno = tips.pivot_table(index="day",columns="time",values="pct",aggfunc="mean")
dia_mayor = resumen["pct_promedio"].idxmax()

print(resumen)
print("------------------------------------------------------------------------------------")
print(dia_turno)
print("------------------------------------------------------------------------------------")
print("El dia con el mayor porcentaje medio de propina es", dia_mayor)

# Verificacion: resumen debe tener 4 filas
print(resumen.shape)

      pct_promedio  num_mesas
day                          
Fri       0.169913         19
Sun       0.166897         76
Thur      0.161276         62
Sat       0.153152         87
------------------------------------------------------------------------------------
time    Dinner     Lunch
day                     
Fri   0.158916  0.188765
Sat   0.153152       NaN
Sun   0.166897       NaN
Thur  0.159744  0.161301
------------------------------------------------------------------------------------
El dia con el mayor porcentaje medio de propina es Fri
(4, 2)
